
# NOTEBOOK: 02_SILVER_PIPELINE
Description: Cleans types and joins Transactions, Cards, Users, and Labels.

In [0]:
# STEP 1: Setup Silver Schema
from pyspark.sql.functions import col, lower, trim, to_date, date_format,when,lit
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

DataFrame[]

In [0]:
# STEP 2: Clean and Cast Tables
spark.table("bronze.transactions_data").withColumn("amount", col("amount").cast("double")).withColumn("date", col("date").cast("timestamp")).write.mode("overwrite").saveAsTable("silver.transactions_data")

spark.table("bronze.cards_data").withColumn("credit_limit", col("credit_limit").cast("double")).write.mode("overwrite").saveAsTable("silver.cards_data")

spark.table("bronze.train_fraud_labels").withColumn("is_fraud", (lower(trim(col("fraud_label"))) == "yes")).write.mode("overwrite").saveAsTable("silver.fraud_labels")


In [0]:
# STEP 3: Join quality check

t = spark.table("silver.transactions_data").alias("t")
c = spark.table("silver.cards_data").alias("c")
u = spark.table("silver.users_data").alias("u")
m = spark.table("silver.mcc_codes").alias("m")
f = spark.table("silver.train_fraud_labels").alias("f")

print("Transactions:", t.count())

print(
    "Card matches:",
    t.join(c, col("t.card_id") == col("c.id"), "left")
     .filter(col("c.id").isNotNull())
     .count()
)

print(
    "User matches:",
    t.join(u, col("t.client_id") == col("u.id"), "left")
     .filter(col("u.id").isNotNull())
     .count()
)

print(
    "MCC matches:",
    t.join(m, col("t.mcc") == col("m.mcc_code"), "left")
     .filter(col("m.mcc_code").isNotNull())
     .count()
)

print(
    "Fraud matches:",
    t.join(f, col("t.id") == col("f.transaction_id"), "left")
     .filter(col("f.transaction_id").isNotNull())
     .count()
)


Transactions: 13305915
Card matches: 13305915
User matches: 13305915
MCC matches: 13305915
Fraud matches: 8914963


In [0]:
from pyspark.sql.functions import col, lower, trim, upper, initcap, date_format, to_date, lit, when#STEP 4: silver.transactions_enriched


transactions_silver_df = spark.table("silver.transactions_data")
cards_silver_df = spark.table("silver.cards_data")
users_silver_df = spark.table("silver.users_data")
mcc_silver_df = spark.table("silver.mcc_codes")
fraud_silver_df = spark.table("silver.train_fraud_labels")

transactions_enriched_df = (
    transactions_silver_df.alias("t")
    .join(
        cards_silver_df.alias("c"),
        col("t.card_id") == col("c.id"),
        "left"
    )
    .join(
        users_silver_df.alias("u"),
        col("t.client_id") == col("u.id"),
        "left"
    )
    .join(
        mcc_silver_df.alias("m"),
        col("t.mcc") == col("m.mcc_code"),
        "left"
    )
    .join(
        fraud_silver_df.alias("f"),
        col("t.id") == col("f.transaction_id"),
        "left"
    )
    .select(
        col("t.id").alias("transaction_id"),
        col("t.date").alias("transaction_timestamp"),
        col("t.client_id"),
        col("t.card_id"),
        col("t.amount"),
        col("t.use_chip"),
        col("t.merchant_id"),
        col("t.merchant_city"),
        col("t.merchant_state"),
        col("t.zip"),
        col("t.mcc"),
        col("t.errors"),
        col("c.card_brand"),
        col("c.card_type"),
        col("c.has_chip"),
        col("c.num_cards_issued"),
        col("c.credit_limit"),
        col("c.acct_open_date"),
        col("c.year_pin_last_changed"),
        col("c.card_on_dark_web"),
        col("m.mcc_description"),
        col("f.fraud_label"),
        col("u.*")
    )
    .withColumn(
        "is_fraud",
        when(lower(col("fraud_label")) == "yes", lit(True)).otherwise(lit(False))
    )
    .withColumn("transaction_date", to_date(col("transaction_timestamp")))
    .withColumn("day_of_week", date_format(col("transaction_timestamp"), "EEEE"))
    .withColumn("merchant_state", upper(trim(col("merchant_state"))))
    .withColumn("merchant_city", initcap(trim(col("merchant_city"))))
)

transactions_enriched_df.write.mode("overwrite").saveAsTable("silver.transactions_enriched")

print("silver.transactions_enriched count:", spark.table("silver.transactions_enriched").count())

silver.transactions_enriched count: 13305915


In [0]:
# Final silver validation

print("silver.transactions_data:", spark.table("silver.transactions_data").count())
print("silver.cards_data:", spark.table("silver.cards_data").count())
print("silver.users_data:", spark.table("silver.users_data").count())
print("silver.mcc_codes:", spark.table("silver.mcc_codes").count())
print("silver.train_fraud_labels:", spark.table("silver.train_fraud_labels").count())
print("silver.transactions_enriched:", spark.table("silver.transactions_enriched").count())

silver.transactions_data: 13305915
silver.cards_data: 6146
silver.users_data: 2000
silver.mcc_codes: 109
silver.train_fraud_labels: 8914963
silver.transactions_enriched: 13305915
